## Retrieval QA
RAG Document Q&A System<br>
Retrieval Augmented Generation with LangChain and Groq<br>

This notebook demonstrates a complete RAG (Retrieval Augmented Generation) <br>
pipeline for answering questions about any PDF document.<br>

What this notebook does:<br>
1. **Loads** a PDF document from a URL
2. **Splits** it into manageable chunks
3. **Embeds** each chunk using HuggingFace's sentence transformers (runs locally)
4. **Stores** embeddings in ChromaDB vector database
5. **Retrieves** relevant chunks based on user questions
6. **Answers** questions using Groq's Llama model

Key concept — RAG:<br>
Instead of asking an LLM questions from its training data alone, <br>
RAG gives the LLM access to specific documents at query time.<br>
This means the LLM answers based on OUR documents, not general knowledge.<br>

Libraries used:<br>
- LangChain — orchestration framework<br>
- ChromaDB — vector database for storing embeddings<br>
- HuggingFace Sentence Transformers — local embedding model<br>
- Groq (Llama 3.3) — LLM for generating answers<br>

##### Imports and setup

In [11]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.getenv("GROQ_API_KEY")
)

##### Load and split PDF

In [12]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

text_splitter = CharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=20, 
    separator="\n"
)
chunks = text_splitter.split_documents(document)
print(f"Total chunks: {len(chunks)}")

Total chunks: 147


##### Embeddings

In [24]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  # small, fast, free, runs locally
)
#The vector uses this 'embeddings' to convert text to numbers.
#Even this 'how it should work' can be configured by adding 'params' according to what we need.
#And then, whatever params we are using, we can even define 'TRUNCATE_INPUT_TOKENS' and 'RETURN_OPTIONS' according to us.
#For more elaboration, see file: '07.1_Embedding Models, Vector Stores, and Retrievers'.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

##### Vector store and QA chain

In [23]:
docsearch = Chroma.from_documents(chunks, embeddings) #docsearch is the vector store here, which stores the 'chunks' and in the form of what it got from 'embeddings'

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", #Takes all retrieved chunks and stuffs them all into one promp
    retriever=docsearch.as_retriever(),
    return_source_documents=True  # let's see sources this time..
)


##### Asking questions

In [19]:
questions = [
    "What is this paper discussing?",
    "What is the purpose of MindGuide?",
    "What LangChain components does MindGuide use?",
    "What mental health problems does MindGuide solve?"
]

for question in questions:
    result = qa.invoke(question)
    print(f"\nQ: {question}")
    print(f"A: {result['result']}")
    print(f"Source: {result['source_documents'][0].page_content[:100]}...")
    print("---")


Q: What is this paper discussing?
A: This text appears to be discussing mental health, specifically the complexities of addressing mental health challenges and the interactions between individuals and a framework called LangChain, which seems to be an AI-powered system designed to facilitate conversations and provide support for mental health issues.
Source: throughout extraordinary demographic businesses and areas. 
However, what makes this situation even ...
---

Q: What is the purpose of MindGuide?
A: The purpose of MindGuide appears to be providing guidance and support to individuals in need, particularly those dealing with issues such as depression and anxiety.
Source: individuals in need of guidance and support in these critical 
areas. MindGuide relies on the capabi...
---

Q: What LangChain components does MindGuide use?
A: According to the provided context, MindGuide uses the following LangChain component: 

1. ChatModel (specifically Chat OpenAI) 

It may use other component

## Notes:
We have other options too other than 'stuff' such as:<br>
map_reduce — for large documents (Good when you have too many chunks to fit in one prompt):<br>
-Step 1: Send each chunk to LLM separately → get mini answer for each<br>
-Step 2: Combine all mini answers → send to LLM again → final answer<br>
refine — iterative approach (Good for nuanced answers that build on each other):<br>
-Step 1: Answer using chunk 1<br>
-Step 2: Refine that answer using chunk 2<br>
-Step 3: Refine again using chunk 3<br>
...until all chunks processed<br>
map_rerank — most precise (Good when you want the single most relevant answer):<br>
-Step 1: Send each chunk to LLM separately → get answer + confidence score<br>
-Step 2: Return the answer with highest confidence score<br>
When to use which:<br>
| Method | To use when: |
| :--- | :--- |
| stuff | Small documents, few chunks – simplest and fastest |
| map_reduce | Large documents, many chunks |
| refine | Need thorough, nuanced answers |
| map_rerank | Need the single most confident answer |

..............................................................................................................................................................................................................<br>
Also, I had some questions, and here's what I got from the internet:<br>
<br>
Q1. On what basis should I select the chunk size? Based on the doc or what?<br>

Answer: There's no perfect formula — it depends on several factors:<br>

Based on our document type:<br>
| Document type | Recommended chunk size |
| :--- | :--- |
| Academic papers | 500-1000 chars — dense, structured content |
| News articles | 300-500 chars — paragraphs are self-contained |
| Legal documents | 1000-2000 chars — context spans long sections |
| Code | 500-1500 chars — functions should stay together |
| Conversations/chat | 200-400 chars — short exchanges |

Based on our use case:<br>
Precise fact retrieval → smaller chunks (200-400) — find exact answers<br>
Conceptual questions → larger chunks (800-1500) — need more context<br>
Summarization → larger chunks — need the full picture<br>

Based on our LLM's token limit:
If we retrieve 4 chunks and each is 1000 chars, that's 4000 chars going into our prompt<br>
Hence, We should make sure chunks × number_retrieved fits within our model's context window<br>

The honest answer is trial and error:<br>
Most people start with chunk_size=500, chunk_overlap=50 and adjust based on whether answers are good or not. <br>
If answers are missing context → increase chunk size. If answers are imprecise → decrease chunk size.<br>

Q2. Here, we did not define embedding params, so when do I know I need to?<br>

Answer:<br>

In the IBM lab we defined:<br>
pythonembed_params = {<br>
    TRUNCATE_INPUT_TOKENS: 3,<br>
    RETURN_OPTIONS: {"input_text": True}<br>
}<br>

On our laptop we didn't define any params for HuggingFace embeddings.<br>
The rule is simple:<br>
We need embed_params when:<br>
Our chunks might be too long for the embedding model's limit → set TRUNCATE_INPUT_TOKENS<br>
We're debugging and want to verify the model received the right text → set RETURN_OPTIONS<br>
We need specific performance settings for the API<br>

We don't need embed_params when:<br>
Using local models like HuggingFace — they handle truncation automatically<br>
Our chunks are small enough to fit within the model's default limits<br>
We're okay with default behaviour<br>


Quick rule of thumb:<br>
IBM/OpenAI API embeddings → usually need params (API has strict limits)<br>
Local HuggingFace embeddings → usually don't need params (handled internally)<br>

## **Let's Try One Ourselves:**
I don't know how this'll go, but let's see:

In [40]:
storyloader = PyPDFLoader("https://primarysite-prod-sorted.s3.amazonaws.com/white-rock/UploadedDocument/ad50ccb2ad9546ada0d227365754d219/charlie-and-the-chocolate-factory-by-roald-dahl.pdf")
story = storyloader.load()

storytext_splitter = CharacterTextSplitter(
    chunk_size=2000,  #Since the story is very large, larger chunk size is needed
    chunk_overlap=200, #Even a larger chunk overlap is needed
    separator="\n"
)

storychunks = storytext_splitter.split_documents(story)
print(f"Total chunks: {len(storychunks)}")

Total chunks: 146


In [41]:
story_vectorstore = Chroma.from_documents(storychunks, embeddings) #docsearch is the vector store here, which stores the 'chunks' and in the form of what it got from 'embeddings'

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", #Takes all retrieved chunks and stuffs them all into one promp
    retriever=story_vectorstore.as_retriever(),
    return_source_documents=True  # let's see sources this time..
)

In [42]:
storyquestions = [
    "What is this story about?",
    "Who are the five children who find the Golden Tickets?",
    "What happens to Violet Beauregarde when she tries the experimental gum?",
    "What is the name of Mr. Wonka's secret workforce?",
    "Compare and contrast Charlie Bucket's upbringing with that of the other four ticket winners (Augustus, Veruca, Violet, and Mike). How do the parenting styles of Mr. and Mrs. Bucket differ from the other parents? In your answer, explain how each child's specific vice is a direct result of their parents' choices, and how this flaws-and-parenting dynamic leads to their inevitable downfall inside the factory."
]

for q in storyquestions:
    result = qa.invoke(q)
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")
    print(f"Source 1: {result['source_documents'][0].page_content[:100]}...")
    # [0] implies to the displaying only the first source/chunk from which the answer was generated. (1 answer can be generated from combining many sources/chunks)
    print("---")


Q: What is this story about?
A: This story appears to be about a young boy named Charlie Bucket and his interactions with his grandfather, Grandpa Joe. The story mentions that the family is struggling with poverty and hunger, but the specific scene provided focuses on Grandpa Joe and Charlie unwrapping a mysterious item, which turns out to be a bar of chocolate, and they both find the situation amusing. The story seems to be a part of a larger narrative, possibly from the book "Charlie and the Chocolate Factory" by Roald Dahl.
Source 1: ‘Very	well,	then.	Here	goes.’	He	tore	off	the	wrapper.
They	both	stared	at	what	lay	underneath.	It	w...
---

Q: Who are the five children who find the Golden Tickets?
A: The text does not mention the names of all five children who find the Golden Tickets. It only mentions that four Golden Tickets have been found, with the headlines "TWO GOLDEN TICKETS FOUND TODAY, ONLY ONE MORE LEFT" and later, it mentions that the main character (who is not explicitly

As we can see, the 5th question failed to provide an answer.<br>
That last question about comparing Charlie vs the other children requires:<br>
Knowledge from multiple chapters<br>
Understanding of character arcs across the whole book<br>
Synthesis of many different parts<br>

But RAG only retrieves 4 chunks by default.<br>
4 chunks from a whole novel is nowhere near enough to answer a question that spans the entire story.<br>

This is a fundamental limitation of RAG:<br>
RAG is great for specific factual questions. It struggles with questions that require understanding the whole document.<br>

However, let's try to increase the retrieved chunks:

In [49]:
# Here, 'qa2' is a RetrievalQA chain object (specifically an instance of the RetrievalQA class).
qa2 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", #Takes all retrieved chunks and stuffs them all into one promp
    retriever=story_vectorstore.as_retriever(search_kwargs={"k": 20}), #Now, we will take the first 20 relevant chunks provided back by Chroma.
    return_source_documents=True  # let's see sources this time..
)

question5 = "Compare and contrast Charlie Bucket's upbringing with that of the other four ticket winners (Augustus, Veruca, Violet, and Mike). How do the parenting styles of Mr. and Mrs. Bucket differ from the other parents? In your answer, explain how each child's specific vice is a direct result of their parents' choices, and how this flaws-and-parenting dynamic leads to their inevitable downfall inside the factory."
result2 = qa2.invoke(question5)
print(f"\nQ: {question5}")
print(f"A: {result2['result']}")
print(f"Source : {result2['source_documents'][0].page_content[:100]}...")


Q: Compare and contrast Charlie Bucket's upbringing with that of the other four ticket winners (Augustus, Veruca, Violet, and Mike). How do the parenting styles of Mr. and Mrs. Bucket differ from the other parents? In your answer, explain how each child's specific vice is a direct result of their parents' choices, and how this flaws-and-parenting dynamic leads to their inevitable downfall inside the factory.
A: The upbringing of Charlie Bucket differs significantly from that of the other four ticket winners, primarily due to the parenting styles of Mr. and Mrs. Bucket. While Charlie's parents are loving, supportive, and teach him the value of humility and hard work, the other parents are often portrayed as overindulgent, neglectful, or pushy.

Augustus Gloop, for example, is raised by parents who constantly give in to his desires, resulting in his gluttony and lack of self-control. His parents' permissiveness and failure to set boundaries lead to Augustus's insatiable appetite and eve

The lesson here is huge, and that is:<br>

The quality of RAG answers depends heavily on how many relevant chunks we retrieve.<br>

Too few → incomplete answers<br>
Too many → risk hitting token limits, slower, more expensive<br>
Finding the right k is part of the art of building RAG systems.<br>

#### Learning Part:
Okay, so now let me just document on how to build a small retrieval system for any DOC based on my experience, and what I learnt:
1. Install of necessary packages needed.
2. Declare an llm_variable to use
3. Take any DOCUMENT
4. Load that document into the DOCUMENT LOADER
5. With the help of the .load() function, load it to the DOCUMENT OBJECT.
6. Define a text splitter by defining chunk size, overlap, and seperator accordingly.
7. Use that text splitter to create chunks of that document.
8. Create and configure a text embedding model.
9. If needed, we may also define embedding parameters as needed. (To be done before Step 8 if needed).
10. Create a vector store of those chunks.
11. Create a retriver of that vector store.
12. Create a RetrievalQA chain object
13. Define a variable for a question
14. Involke that question in the format: retrievalQAobject.invoke(question)